# ACI — Serveur d'inférence vLLM sur Google Colab (1× NVIDIA T4 16 Go)

Ce notebook déploie un serveur **vLLM compatible OpenAI** (`/v1/chat/completions`) calibré pour les contraintes matérielles de la **NVIDIA Tesla T4 (16 Go VRAM, architecture Turing `sm_75`)** :
- `--dtype half` (`float16`, car la T4 ne supporte pas `bfloat16` natif)
- Quantification **AWQ 4-bit** (`Qwen2.5-7B-Instruct-AWQ` par défaut, ~6 Go VRAM)
- Tunnel HTTPS gratuit **Cloudflare Quick Tunnel** (`cloudflared`, aucun compte requis) pour relier votre **Inference Gateway** locale ACI à Colab.

> **Prérequis** : dans Colab, sélectionnez *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# 1. Vérifier la présence de la GPU T4 et installer vLLM + cloudflared
!nvidia-smi
!pip install -q vllm>=0.6.3 httpx
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 2. Configuration du tier de modèle pour 1x T4 (16 Go VRAM)
MODEL_TIERS = {
    "small": "Qwen/Qwen2.5-3B-Instruct-AWQ",
    "medium": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "large": "Qwen/Qwen2.5-14B-Instruct-AWQ",  # ~10 Go VRAM : réduire MAX_MODEL_LEN à 4096 si besoin
}

SELECTED_TIER = "medium"  # Choisir: "small", "medium" ou "large"
MODEL_NAME = MODEL_TIERS[SELECTED_TIER]
API_KEY = "aci-colab-dev-key"
PORT = 8000
MAX_MODEL_LEN = 4096 if SELECTED_TIER == "large" else 8192
GPU_MEMORY_UTILIZATION = 0.85

print(f"Tier={SELECTED_TIER} | Modèle={MODEL_NAME} | max_model_len={MAX_MODEL_LEN}")

In [ ]:
# 3. Démarrer vLLM en arrière-plan (optimisé T4 : --dtype half --quantization awq --enforce-eager)
import subprocess
import time
import httpx

vllm_cmd = [
    "vllm", "serve", MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--dtype", "half",
    "--quantization", "awq",
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--api-key", API_KEY,
    "--enforce-eager",
]

log_file = open("/tmp/vllm_aci.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)
print("Démarrage de vLLM (téléchargement + chargement des poids sur T4)...")

ready = False
for i in range(90):
    try:
        r = httpx.get(f"http://127.0.0.1:{PORT}/health", timeout=2.0)
        if r.status_code == 200:
            ready = True
            print(f"vLLM prêt sur le port {PORT} après {i * 5}s !")
            break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    print("vLLM n'a pas répondu à temps. Dernières lignes de /tmp/vllm_aci.log :")
    !tail -n 40 /tmp/vllm_aci.log

In [ ]:
# 4. Ouvrir le tunnel Cloudflare vers http://localhost:8000
import re
import subprocess
import time

tunnel_log = "/tmp/cloudflared.log"
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(1)
    text = open(tunnel_log).read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=== Copiez ces variables dans votre fichier local .env ===")
    print(f"ACI_INFERENCE_MODE=auto")
    print(f"ACI_VLLM_BASE_URL={public_url}/v1")
    print(f"ACI_VLLM_API_KEY={API_KEY}")
else:
    print("Impossible de récupérer l'URL du tunnel. Voir /tmp/cloudflared.log :")
    !cat /tmp/cloudflared.log

In [ ]:
# 5. Test rapide OpenAI-compatible + extraction JSON structurée sur T4
import json
import httpx

resp = httpx.post(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL_NAME,
        "temperature": 0.0,
        "max_tokens": 256,
        "response_format": {"type": "json_object"},
        "messages": [
            {
                "role": "system",
                "content": "Tu es un extracteur financier. Réponds uniquement en JSON valide avec les clés revenue_mad et page.",
            },
            {
                "role": "user",
                "content": "Extrait (Resultat.pdf, p. 2) : Chiffre d'affaires 12 000 000 MAD, Résultat net 850 000 MAD.",
            },
        ],
    },
    timeout=30.0,
)
resp.raise_for_status()
data = resp.json()
print(json.dumps(data["choices"][0]["message"], indent=2, ensure_ascii=False))
print("Usage:", data.get("usage"))

In [ ]:
# 6. Monitoring VRAM / Utilisation GPU T4
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv